# Dyck Hidden State Evaluation: Train, Probe, Compression

This notebook is runnable end-to-end with the current `src/hse` implementation. By default it runs a small smoke experiment over both Dyck settings and all four model families:

- Dyck no noise
- Dyck 50% noise
- RNN, LSTM, Transformer, and official Mamba when `mamba-ssm` is available

The default values are intentionally small so `Run All` finishes quickly. For the real experiment, increase `TRAINING_STEPS`, `EXTRACT_EXAMPLES`, `BATCH_SIZE`, and `SEEDS` in Section 1.

## 0. Setup

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

RESULTS = ROOT / "results" / "dyck_all_models_notebook"
RESULTS.mkdir(parents=True, exist_ok=True)

ROOT, RESULTS

In [ ]:
import numpy as np
import pandas as pd
import torch

from hse.analysis.compression import run_compression_probes
from hse.analysis.probes import load_probe_data, run_sufficient_statistic_probes
from hse.models import OfficialMambaUnavailableError, build_model
from hse.tasks.dyck import DyckConfig, DyckSampler
from hse.utils import evaluate_causal_lm, extract_hidden_states, save_json, train_causal_lm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

## 1. Settings

The notebook defaults to a quick smoke run. For the real run, use something like:

```python
TRAINING_STEPS = 10_000
BATCH_SIZE = 128
EXTRACT_EXAMPLES = 50_000
SEEDS = [0, 1, 2]
```

Everything else can stay the same.

In [ ]:
DYCK_SETTINGS = {
    "dyck_no_noise": dict(
        dyck_pairs=24,
        total_length=48,
        seq_len=48,
        repeat_prob=1.0,
        n_tasks=512,
        prefix_probe_max_len=7,
    ),
    "dyck_50_noise": dict(
        dyck_pairs=10,
        total_length=20,
        seq_len=48,
        repeat_prob=0.5,
        n_tasks=512,
        prefix_probe_max_len=7,
    ),
}

MAMBA_SSM_AVAILABLE = importlib.util.find_spec("mamba_ssm") is not None
RUN_MAMBA_LIKE_FALLBACK_WHEN_OFFICIAL_UNAVAILABLE = True

MODEL_SPECS = {
    "rnn": dict(layers=3, emb_dim=128, hidden_dim=256, state_kind="h"),
    "lstm": dict(layers=3, emb_dim=128, hidden_dim=128, state_kind="c"),
    "transformer": dict(layers=3, emb_dim=128, hidden_dim=128, n_heads=4, ffn_dim=512, state_kind="h"),
}

if MAMBA_SSM_AVAILABLE:
    MODEL_SPECS["mamba"] = dict(
        layers=3,
        emb_dim=128,
        hidden_dim=128,
        state_dim=16,
        expansion_factor=2,
        require_official_mamba=True,
        state_kind="h",
    )
elif RUN_MAMBA_LIKE_FALLBACK_WHEN_OFFICIAL_UNAVAILABLE:
    MODEL_SPECS["mamba_like"] = dict(
        layers=3,
        emb_dim=128,
        hidden_dim=128,
        state_dim=16,
        expansion_factor=2,
        state_kind="h",
    )

# Direct-run defaults. These are deliberately small.
SEEDS = [0]
TRAINING_STEPS = 5
BATCH_SIZE = 16
EVAL_EVERY = 5
EXTRACT_EXAMPLES = 64
EXTRACT_BATCH_SIZE = 32
LR = 3e-4

pd.DataFrame(DYCK_SETTINGS).T

## 2. Mamba Availability Check

If the official `mamba-ssm` package is installed, this notebook runs `model_name="mamba"` with `require_official_mamba=True`. On this Windows environment, official `mamba-ssm` is not installed, so the notebook can optionally run `model_name="mamba_like"` as a smoke-test fallback. Treat `mamba_like` as an engineering placeholder, not as official Mamba evidence.

In [ ]:
mamba_ssm_available = importlib.util.find_spec("mamba_ssm") is not None
test_model_name = "mamba" if mamba_ssm_available else "mamba_like"
test_mamba = build_model(
    test_model_name,
    vocab_size=12,
    emb_dim=16,
    hidden_dim=16,
    layers=1,
    require_official_mamba=mamba_ssm_available,
)
x = torch.randint(0, 12, (2, 8))
with torch.no_grad():
    logits = test_mamba(x)
    states = test_mamba.extract_states(x)

{
    "mamba_ssm_available": mamba_ssm_available,
    "requested_model": test_model_name,
    "model_class": type(test_mamba).__name__,
    "uses_mamba_ssm": getattr(test_mamba, "uses_mamba_ssm", None),
    "logits_shape": tuple(logits.shape),
    "states_shape": tuple(states.shape),
}

## 3. Train And Extract

In [ ]:
def run_one(setting_name, setting_kwargs, model_name, model_spec, seed):
    run_dir = RESULTS / setting_name / f"{model_name}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    task_cfg = DyckConfig(**setting_kwargs, device=DEVICE)
    sampler = DyckSampler(task_cfg, seed=seed)
    model_kwargs = {k: v for k, v in model_spec.items() if k != "state_kind"}

    torch.manual_seed(seed)
    model = build_model(model_name, vocab_size=sampler.vocab_size, **model_kwargs).to(DEVICE)

    config_record = {
        "setting_name": setting_name,
        "task": {**setting_kwargs, "device": DEVICE},
        "model_name": model_name,
        "model": model_spec,
        "seed": seed,
        "training_steps": TRAINING_STEPS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "device": DEVICE,
    }
    save_json(config_record, run_dir / "config.json")

    train_log = train_causal_lm(
        model=model,
        sampler=sampler,
        steps=TRAINING_STEPS,
        batch_size=BATCH_SIZE,
        lr=LR,
        run_dir=run_dir,
        eval_every=EVAL_EVERY,
        device=DEVICE,
    )
    eval_metrics = evaluate_causal_lm(model=model, sampler=sampler, batch_size=BATCH_SIZE, device=DEVICE)
    save_json({"train": train_log, "eval": eval_metrics}, run_dir / "metrics.json")

    hidden, labels = extract_hidden_states(
        model=model,
        sampler=sampler,
        state_kind=model_spec.get("state_kind", "h"),
        layer=-1,
        num_examples=EXTRACT_EXAMPLES,
        batch_size=EXTRACT_BATCH_SIZE,
        max_prefix_len=task_cfg.prefix_probe_max_len,
        device=DEVICE,
        run_dir=run_dir,
    )
    return run_dir, eval_metrics, tuple(hidden.shape), labels.shape

In [ ]:
run_rows = []

for setting_name, setting_kwargs in DYCK_SETTINGS.items():
    for model_name, model_spec in MODEL_SPECS.items():
        for seed in SEEDS:
            run_dir, eval_metrics, hidden_shape, labels_shape = run_one(
                setting_name, setting_kwargs, model_name, model_spec, seed
            )
            run_rows.append({
                "setting": setting_name,
                "model": model_name,
                "seed": seed,
                "run_dir": str(run_dir.relative_to(ROOT)),
                "loss": eval_metrics["loss"],
                "accuracy": eval_metrics["accuracy"],
                "dyck_accuracy": eval_metrics["dyck_accuracy"],
                "hidden_shape": hidden_shape,
                "labels_shape": labels_shape,
            })

runs_df = pd.DataFrame(run_rows)
runs_df.to_csv(RESULTS / "runs.csv", index=False)
runs_df

## 4. Probes And Compression

The probe layer uses sklearn if explicitly enabled with `HSE_USE_SKLEARN=1`; otherwise it uses the built-in torch/numpy fallback. This avoids local NumPy/scipy ABI issues.

In [ ]:
def labels_path_for(run_dir):
    parquet = run_dir / "labels.parquet"
    csv = run_dir / "labels.csv"
    return parquet if parquet.exists() else csv


def analyze_one(run_dir, seed):
    X, labels = load_probe_data(run_dir / "hidden_states.pt", labels_path_for(run_dir))
    probe_results = run_sufficient_statistic_probes(X, labels, seed=seed)
    compression_rows, compression_summary = run_compression_probes(X, labels, seed=seed)

    probe_dir = run_dir / "probes"
    probe_dir.mkdir(exist_ok=True)
    compression_rows.to_csv(probe_dir / "compression_probe_rows.csv", index=False)

    summary = {
        "left_r2": probe_results["left"]["r2"],
        "right_r2": probe_results["right"]["r2"],
        "height_r2": probe_results["height"]["r2"],
        "height_mae": probe_results["height"]["mae"],
        **probe_results["geometry"],
        **compression_summary,
    }
    save_json(summary, probe_dir / "summary.json")
    return summary

In [ ]:
analysis_rows = []

for row in run_rows:
    run_dir = ROOT / row["run_dir"]
    summary = analyze_one(run_dir, seed=row["seed"])
    analysis_rows.append({
        "setting": row["setting"],
        "model": row["model"],
        "seed": row["seed"],
        **summary,
    })

analysis_df = pd.DataFrame(analysis_rows)
analysis_df.to_csv(RESULTS / "probe_summary.csv", index=False)
analysis_df

## 5. Interpretation Checklist

For the smoke run, the numbers only verify that the pipeline runs. They are not research evidence. For the full run, check:

- `dyck_accuracy`: task performance on Dyck target positions.
- `height_r2`: linear decodability of stack height.
- `cos_height_left_minus_right`: whether the learned height direction matches `w_left - w_right`.
- `relevant_retention`: how much sufficient-statistic information is retained.
- `irrelevant_forgetting`: how much noise/prefix detail is discarded.

A strong compression story means high relevant retention and high irrelevant forgetting, not just low information overall.